> **Note on this notebook — why it carries no outputs**
>
> This was the first attempt at retraining the Temporal Fusion Transformer, run on a
> Kaggle GPU session that was interrupted before it finished. It is kept as a record of
> that attempt rather than as a result.
>
> **Why it has not been re-run:** it trains for up to 40 epochs at a learning rate of
> 0.0003. On the GPU available for this project that is roughly eighteen hours, beyond
> what a single session allows. Reducing the epoch count to fit would no longer be the
> run being recorded. More importantly, 0.0003 is the learning rate this project later
> identified as the cause of the transformer's poor early results, so a completed run
> here would reproduce a configuration already shown to be wrong.
>
> **What replaced it:** `15_tft_retrain_colab.ipynb` is the retrain that produced the
> final model, and `notebooks/16_tft_evaluation.ipynb` loads the resulting checkpoint and
> reproduces the reported test MAE of 1.247 on CPU. That notebook is executed and the
> checkpoint is included in `best_model/`.

> **Note on outputs**
>
> **Superseded.** This was the first attempt at retraining the TFT with tuned hyperparameters. The Kaggle run was cancelled part way through, so it never converged and the results were not usable.
>
> **Use instead:** notebook 15 (15_tft_retrain_colab.ipynb), which is the retrain that actually completed and produced the final model. Kept here only to document what was tried.

# Notebook 3 retrain - TFT with better hyperparams

Last run got TFT MAE = 1.53 which is worse than LSTM (1.39).
Trying again with bigger model and more epochs to see if we can beat LSTM.

Changes:
- hidden_size 64 -> 128
- attention heads 4 -> 8  
- learning rate 1e-3 -> 3e-4 (lower lr, longer training)
- dropout 0.2 -> 0.15
- max_epochs 30 -> 40
- batch_size 128 (keeping this, batch=64 was too slow on Kaggle)

Run as commit not draft. Should take 3-5 hrs.

In [ ]:
!pip install -q pytorch-lightning==2.6.4 pytorch-forecasting==1.7.0

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# pytorch 2.6 needs this otherwise checkpoint load fails
torch.serialization.add_safe_globals([GroupNormalizer])

pl.seed_everything(42)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())

In [ ]:
# find where kaggle put the data
def find_csv_dir(root='/kaggle/input'):
    for dirpath, _, files in os.walk(root):
        if 'jena_germany_2000_2009_hourly.csv' in files:
            return dirpath + '/'
    return None

DATA_DIR = find_csv_dir()
print('found data at:', DATA_DIR)

WORK = '/kaggle/working/'
os.makedirs(WORK + 'models', exist_ok=True)
os.makedirs(WORK + 'processed', exist_ok=True)
os.makedirs(WORK + 'plots', exist_ok=True)

DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'

## Load the 5 city CSVs

In [ ]:
city_meta = {
    'jena_germany_2000_2009_hourly.csv':    ('Jena',     50.9227, 11.5865),
    'london_uk_2000_2009_hourly.csv':       ('London',   51.5074, -0.1278),
    'newyork_usa_2000_2009_hourly.csv':     ('New York', 40.7128, -74.0060),
    'sydney_australia_2000_2009_hourly.csv':('Sydney',  -33.8688, 151.2093),
    'tokyo_japan_2000_2009_hourly.csv':     ('Tokyo',    35.6762, 139.6503),
}

# rename open-meteo column names to something shorter
rename_map = {
    'temperature_2m': 'temperature',
    'relative_humidity_2m': 'humidity',
    'dew_point_2m': 'dew_point',
    'precipitation': 'precipitation',
    'rain': 'rain',
    'wind_speed_10m': 'wind_speed',
    'wind_direction_10m': 'wind_direction',
    'wind_gusts_10m': 'wind_gusts',
    'pressure_msl': 'pressure_msl',
    'surface_pressure': 'surface_pressure',
    'cloud_cover': 'cloud_cover',
    'cloud_cover_low': 'cloud_cover_low',
    'cloud_cover_mid': 'cloud_cover_mid',
    'cloud_cover_high': 'cloud_cover_high',
    'shortwave_radiation': 'shortwave_radiation',
    'direct_radiation': 'direct_radiation',
    'vapour_pressure_deficit': 'vapour_pressure_deficit',
    'wet_bulb_temperature_2m': 'wet_bulb_temp',
    'total_column_integrated_water_vapour': 'water_vapour',
    'soil_temperature_0_to_7cm': 'soil_temperature',
    'et0_fao_evapotranspiration': 'evapotranspiration',
}

def clean_col(c):
    # strip stuff like " (°C)" from column names
    c = re.sub(r'\s*\([^)]*\)', '', c)
    return c.strip()

all_dfs = []
for fname, (city, lat, lon) in city_meta.items():
    fp = DATA_DIR + fname
    d = pd.read_csv(fp, skiprows=3, parse_dates=['time'])
    d.columns = [clean_col(c) for c in d.columns]
    # sometimes pandas adds .1 .2 etc for dupe columns, drop those
    d = d.loc[:, ~d.columns.str.contains(r'\.')]
    d = d.loc[:, ~d.columns.duplicated()]
    d = d.rename(columns=rename_map)
    # keep only columns we care about
    keep = ['time'] + [c for c in rename_map.values() if c in d.columns]
    d = d[keep]
    d['city'] = city
    d['lat'] = lat
    d['lon'] = lon
    all_dfs.append(d)
    print(city, d.shape)

master = pd.concat(all_dfs, ignore_index=True)
print('total rows:', len(master))
master.head()

In [ ]:
# quick check on NaNs
master.isna().sum().sort_values(ascending=False).head(10)

## Feature engineering
Adding hour/month/dayofyear cyclical encodings + wind u/v components

In [ ]:
df = master.sort_values(['city', 'time']).reset_index(drop=True)

df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['dayofyear'] = df['time'].dt.dayofyear

# cyclical encoding so model knows hour 23 is close to hour 0
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)
df['dayofyear_sin'] = np.sin(2*np.pi*df['dayofyear']/365)
df['dayofyear_cos'] = np.cos(2*np.pi*df['dayofyear']/365)

# wind components - splitting direction+speed into u/v
theta = np.deg2rad(df['wind_direction'])
df['wind_u'] = -df['wind_speed'] * np.sin(theta)
df['wind_v'] = -df['wind_speed'] * np.cos(theta)

print('shape after FE:', df.shape)
df.head(2)

In [ ]:
# NaN cleanup - forward fill per city then zero fill anything left
df['temperature'] = df.groupby('city')['temperature'].transform(lambda x: x.ffill().bfill())

for col in df.columns:
    if col in ['time', 'city', 'lat', 'lon']:
        continue
    if df[col].dtype == 'O':
        continue
    df[col] = df.groupby('city')[col].transform(lambda x: x.ffill().bfill())
    df[col] = df[col].fillna(0)

# time_idx is just a counter per city
df['time_idx'] = df.groupby('city').cumcount()

# drop any rows still missing target
df = df.dropna(subset=['temperature']).reset_index(drop=True)

print('final rows:', len(df))
print('NaN remaining:', df.isna().sum().sum())

## Define features for TFT

In [ ]:
TARGET = 'temperature'

static_categoricals = ['city']
static_reals = ['lat', 'lon']

# stuff we always know in advance (calendar features)
time_varying_known_reals = [
    'time_idx',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dayofyear_sin', 'dayofyear_cos',
]

# weather variables - only known up to present
candidate_unknowns = [
    'temperature', 'humidity', 'dew_point',
    'precipitation', 'rain',
    'wind_speed', 'wind_direction', 'wind_gusts',
    'wind_u', 'wind_v',
    'pressure_msl', 'surface_pressure',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'shortwave_radiation', 'direct_radiation',
    'vapour_pressure_deficit', 'wet_bulb_temp',
    'water_vapour', 'soil_temperature', 'evapotranspiration',
]
time_varying_unknown_reals = [c for c in candidate_unknowns if c in df.columns]

print('using', len(time_varying_unknown_reals), 'unknown features')
print(time_varying_unknown_reals)

In [ ]:
# train/val/test split - using time-based split to avoid leakage
MAX_TIME_IDX = df['time_idx'].max()
TRAIN_END = int(MAX_TIME_IDX * 0.70)
VAL_END   = int(MAX_TIME_IDX * 0.85)

ENCODER_LEN = 168  # 1 week lookback
DECODER_LEN = 24   # forecast next 24 hours

print('max time_idx:', MAX_TIME_IDX)
print('train end:', TRAIN_END)
print('val end:', VAL_END)

## Build TimeSeriesDataSet

In [ ]:
training = TimeSeriesDataSet(
    df[df['time_idx'] <= TRAIN_END],
    time_idx='time_idx',
    target=TARGET,
    group_ids=['city'],
    min_encoder_length=ENCODER_LEN // 2,
    max_encoder_length=ENCODER_LEN,
    min_prediction_length=1,
    max_prediction_length=DECODER_LEN,
    static_categoricals=static_categoricals,
    static_reals=static_reals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(groups=['city']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df[df['time_idx'] <= VAL_END],
    predict=False, stop_randomization=True
)

testing = TimeSeriesDataSet.from_dataset(
    training, df,
    predict=False, stop_randomization=True
)

# batch=128 - bigger than 64 because batch=64 made each epoch take 40+ min on T4
BATCH = 128
train_loader = training.to_dataloader(train=True, batch_size=BATCH, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
test_loader = testing.to_dataloader(train=False, batch_size=BATCH, num_workers=0)

print('train batches:', len(train_loader))
print('val batches:', len(val_loader))
print('test batches:', len(test_loader))

## Build TFT (tuned)

In [ ]:
model = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=3e-4,
    hidden_size=128,
    attention_head_size=8,
    dropout=0.15,
    hidden_continuous_size=64,
    output_size=7,
    loss=QuantileLoss(),
    optimizer='adamw',
    reduce_on_plateau_patience=4,
)

n_params = sum(p.numel() for p in model.parameters())
print('TFT params:', n_params)

## Train

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    min_delta=1e-4,
)

ckpt = ModelCheckpoint(
    dirpath=WORK + 'models/',
    filename='tft_tuned_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = pl.Trainer(
    max_epochs=40,
    accelerator=DEVICE,
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_monitor, ckpt],
    enable_progress_bar=True,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print('done training')
print('best ckpt:', ckpt.best_model_path)

## Evaluate on test set

In [ ]:
best_model = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)

predictions = best_model.predict(test_loader, return_y=True, return_x=True, mode='prediction')
preds = predictions.output.cpu().numpy()
actuals = predictions.y[0].cpu().numpy()

print('preds shape:', preds.shape)
print('actuals shape:', actuals.shape)

# mask out any NaN to avoid breaking sklearn
pf = preds.flatten()
af = actuals.flatten()
mask = np.isfinite(pf) & np.isfinite(af)

mae = mean_absolute_error(af[mask], pf[mask])
rmse = np.sqrt(mean_squared_error(af[mask], pf[mask]))
r2 = r2_score(af[mask], pf[mask])

In [ ]:
print('TFT (tuned) results')
print('-------------------')
print('MAE: ', round(mae, 4))
print('RMSE:', round(rmse, 4))
print('R2:  ', round(r2, 4))
print()
print('comparison:')
print('  LSTM           :', 1.3928)
print('  TFT (old, 29ep):', 1.5260)
print('  TFT (tuned)    :', round(mae, 4))

if mae < 1.3928:
    print('beat LSTM!')
else:
    print('still worse than LSTM by', round((mae - 1.3928), 4))

np.save(WORK + 'processed/tft_tuned_preds.npy', preds)
np.save(WORK + 'processed/tft_tuned_actuals.npy', actuals)
print('saved preds')

## Per-city breakdown

In [ ]:
city_indices = predictions.x['groups'].cpu().numpy().flatten()
city_list = sorted(df['city'].unique())

rows = []
for i, city in enumerate(city_list):
    m = city_indices == i
    if not m.any():
        continue
    a = actuals[m].flatten()
    p = preds[m].flatten()
    valid = np.isfinite(a) & np.isfinite(p)
    rows.append({
        'City': city,
        'MAE': round(mean_absolute_error(a[valid], p[valid]), 4),
        'RMSE': round(float(np.sqrt(mean_squared_error(a[valid], p[valid]))), 4),
        'R2': round(r2_score(a[valid], p[valid]), 4),
    })

city_df = pd.DataFrame(rows)
print(city_df.to_string(index=False))

city_df.to_csv(WORK + 'processed/tft_tuned_per_city.csv', index=False)

# also append to overall results csv
row = {
    'model': 'TFT-Tuned',
    'city': 'All 5',
    'MAE': round(mae, 4),
    'RMSE': round(rmse, 4),
    'R2': round(r2, 4),
    'seq_len': ENCODER_LEN,
    'pred_steps': DECODER_LEN,
    'features': len(time_varying_unknown_reals),
}
results_path = WORK + 'processed/model_results.csv'
if os.path.exists(results_path):
    res_df = pd.concat([pd.read_csv(results_path), pd.DataFrame([row])], ignore_index=True)
else:
    res_df = pd.DataFrame([row])
res_df.to_csv(results_path, index=False)
print('saved results')

## Plot MAE by horizon

In [ ]:
horizon_mae = []
for h in range(DECODER_LEN):
    a = actuals[:, h]
    p = preds[:, h]
    m = np.isfinite(a) & np.isfinite(p)
    horizon_mae.append(mean_absolute_error(a[m], p[m]))

plt.figure(figsize=(10, 5))
plt.plot(range(1, DECODER_LEN+1), horizon_mae, marker='o', label='TFT tuned')
plt.axhline(mae, color='green', linestyle='--', label='avg MAE = ' + str(round(mae, 3)))
plt.axhline(1.5260, color='orange', linestyle='-.', label='old TFT (1.53)')
plt.axhline(1.3928, color='red', linestyle=':', label='LSTM (1.39)')
plt.xlabel('hours ahead')
plt.ylabel('MAE (C)')
plt.title('TFT tuned - MAE by forecast horizon')
plt.xticks(range(1, DECODER_LEN+1))
plt.legend()
plt.tight_layout()
plt.savefig(WORK + 'plots/tft_tuned_horizon.png', dpi=150)
plt.show()

print('done')